In [ ]:
import torch

# checking for GPU connectivity
print(torch.cuda.is_available())

import pandas as pd

df = pd.read_parquet('eval_retrieval_pairs.parquet')
df.head()

False


,id,domain,stratum,year,title,abstract,social_post,model
0,W3128646645,biomedical_health,high_cited_recent,2021,Global Cancer Statistics 2020: GLOBOCAN Estima...,This article provides an update on the global ...,Just saw that 2020 had ~19.3M new cancer cases...,gpt-oss-120b
1,W3118615836,biomedical_health,high_cited_recent,2021,The PRISMA 2020 statement: an updated guidelin...,The Preferred Reporting Items for Systematic r...,Just learned the PRISMA guidelines got a big o...,gpt-oss-120b
2,W2889646458,biomedical_health,high_cited_recent,2018,Global cancer statistics 2018: GLOBOCAN estima...,This article provides a status report on the g...,Did you know 2018 saw ~18 million new cancer c...,gpt-oss-120b
3,W2891378911,biomedical_health,high_cited_recent,2018,PRISMA Extension for Scoping Reviews (PRISMA-S...,"Scoping reviews, a type of knowledge synthesis...",Just found out the PRISMA-ScR guide for scopin...,gpt-oss-120b
4,W3008827533,biomedical_health,high_cited_recent,2020,Clinical Characteristics of Coronavirus Diseas...,"BACKGROUND: Since December 2019, when coronavi...","Turns out the average COVID patient was 47 yo,...",gpt-oss-120b


In [ ]:

# loading dataframe

import pandas as pd

df = pd.read_csv('cleaned_metadata.csv')

# dropping unnamed columns (indexes and stuff created on downloads)
df = df.loc[:, ~df.columns.str.contains('^Unnamed:')]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20500 entries, 0 to 20499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 20500 non-null  object
 1   title              20500 non-null  object
 2   authors            20500 non-null  object
 3   abstract           20500 non-null  object
 4   date               20500 non-null  object
 5   year               20500 non-null  int64 
 6   doi                20499 non-null  object
 7   url                20500 non-null  object
 8   venue              20334 non-null  object
 9   cited_by           20500 non-null  int64 
 10  concepts           20500 non-null  object
 11  domain             20500 non-null  object
 12  stratum            20500 non-null  object
 13  embedding_string   20500 non-null  object
 14  bm25_index_string  20500 non-null  object
dtypes: int64(2), object(13)
memory usage: 2.3+ MB


In [ ]:
# generating a balanced domain dataframe to produce a balanced evaluation set
from math import inf


domain_list = df['domain'].unique()
min_len = inf

for domain in domain_list:
  seg_length = len(df[df['domain'] == domain])
  print(domain,' : ',seg_length)
  min_len = min(min_len,seg_length)

balanced_df = pd.DataFrame()

for domain in domain_list:
  balanced_df = pd.concat([balanced_df,df[df['domain'] == domain][:min_len - 450]])

balanced_df = balanced_df[:1800]
balanced_df.shape

biomedical_health  :  9000
nutrition_food_science  :  6000
neuroscience_psychology  :  3000
biology_genetics  :  1500
chemistry_toxicology  :  1000


(1800, 15)

In [ ]:
# =========================
# INSTALLS
# =========================
!pip install cerebras-cloud-sdk -q

import os
import re
import time
import pandas as pd
from cerebras.cloud.sdk import Cerebras

# ── CONFIG ────────────────────────────────────────────────────────────────────
CEREBRAS_API_KEY = "csk-j5m42fhnenkhcjd63cwmd3peeytw694nxjmnvdjd9rm8twkn"
MODEL            = "gpt-oss-120b"
REQUESTS_PER_MIN = 28
OUTPUT_PATH      = "eval_retrieval_pairs.parquet"
# ─────────────────────────────────────────────────────────────────────────────

client   = Cerebras(api_key=CEREBRAS_API_KEY)
interval = 60.0 / REQUESTS_PER_MIN

SYSTEM_PROMPT = """You are a social media post writer. You output ONLY the final post — no thinking, no preamble, no "Let's craft", no "Here's a tweet", no explanation before or after. Just the raw post text.

RULES:
- Write like a real person on Twitter/X, Reddit, or Instagram — not a science journalist
- Translate jargon into plain everyday words (e.g. "myocardial infarction" → "heart attack", "machine learning model" → "AI", "statistically significant" → "actually proven", "randomized controlled trial" → "real study")
- The post must contain a concrete factual claim drawn from the paper
- Use casual language, abbreviations, maybe 1-2 relevant emojis, and 1-3 hashtags
- Keep it under 240 characters where possible, but don't sacrifice the claim
- Do NOT mention the paper, authors, journal, or say "new study" — write as if sharing something you just learned
- Output ONLY the post text. No explanation, no quotes around it, no preamble."""

USER_TEMPLATE = """Paper title: {title}

Abstract: {abstract}

Write the social media post:"""


def parse_retry_after(error_message: str) -> float:
    match = re.search(r'(?:(\d+)m)?(\d+(?:\.\d+)?)s', str(error_message))
    if match:
        return float(match.group(1) or 0) * 60 + float(match.group(2)) + 2
    return 30.0


def generate_post(title: str, abstract: str, retries: int = 4) -> str | None:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_TEMPLATE.format(
                        title=title,
                        abstract=abstract[:1200]
                    )}
                ],
                temperature=0.85,
                max_tokens=1024,
                top_p=0.95,
            )
            content = response.choices[0].message.content
            if not content:
                content = response.choices[0].message.reasoning
            if not content:
                raise ValueError("Both content and reasoning are empty")

            # extract last non-empty line — reasoning models write the post last
            lines = [l.strip() for l in content.strip().splitlines() if l.strip()]
            post  = next(
                (l for l in reversed(lines)
                 if not l.lower().startswith(("let", "here", "we need", "craft", "sure", "okay", "alright"))),
                lines[-1]
            )

            # strip surrounding quotes if model wrapped the post in them
            post = post.strip('"').strip("'").strip()
            return post

        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "rate" in err_str.lower():
                wait = parse_retry_after(err_str)
                print(f"  [Rate limit] Sleeping {wait:.0f}s...")
                time.sleep(wait)
            else:
                wait = 2 ** attempt
                print(f"  [attempt {attempt+1}] Error: {e} — retrying in {wait}s")
                time.sleep(wait)
    return None


# ── PREVIEW: test on 5 rows before full run ───────────────────────────────────
print("=" * 60)
print("PREVIEW — first 5 posts. Interrupt if quality is bad.")
print("=" * 60)
for _, row in balanced_df.head(5).iterrows():
    post = generate_post(row["title"], row["abstract"])
    print(f"\nTitle:  {row['title'][:80]}...")
    print(f"Domain: {row['domain']}")
    print(f"Post:   {post}")
    print("-" * 60)
    time.sleep(interval)

print("\nIf posts look good, run the next cell to process all rows.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 2.8 MB/s eta 0:00:00
PREVIEW — first 5 posts. Interrupt if quality is bad.

Title:  Global Cancer Statistics 2020: GLOBOCAN Estimates of Incidence and Mortality Wor...
Domain: biomedical_health
Post:   Did you know breast cancer just overtook lung cancer as the most common diagnosis? 2.3 M new cases in 2020 vs 2.2 M lung. Still, lung kills the most—1.8 M deaths. 😮 #cancerstats #health
------------------------------------------------------------

Title:  The PRISMA 2020 statement: an updated guideline for reporting systematic reviews...
Domain: biomedical_health
Post:   Just learned PRISMA got a 2020 makeover – 27 checklist items, fresh flow diagrams & an abstract guide to make systematic reviews way clearer. 🙌 #research #OpenScience
------------------------------------------------------------

Title:  Global cancer statistics 2018: GLOBOCAN estimates of incidence and mortality wor...
Domain: biomedical_health
Post:   Wow, 2018 ha

In [ ]:
from google.colab import files

# ── MAIN LOOP — run this cell only if preview looks good ─────────────────────
if os.path.exists(OUTPUT_PATH):
    existing = pd.read_parquet(OUTPUT_PATH)
    done_ids = set(existing["id"].tolist())
    print(f"Resuming — {len(done_ids)} rows already done.")
else:
    existing = pd.DataFrame()
    done_ids = set()

todo = balanced_df[~balanced_df["id"].isin(done_ids)].copy()
print(f"Rows to process: {len(todo)} | Model: {MODEL} | {REQUESTS_PER_MIN} RPM")
print(f"Estimated time: ~{len(todo) / REQUESTS_PER_MIN:.0f} minutes")

results = []

for i, (_, row) in enumerate(todo.iterrows()):
    post = generate_post(row["title"], row["abstract"])

    # print every post so you can monitor quality live
    if (i + 1) % 10 == 0:
      print(f"[{i+1}/{len(todo)}] {post}")

    results.append({
        "id":          row["id"],
        "domain":      row["domain"],
        "stratum":     row["stratum"],
        "year":        row["year"],
        "title":       row["title"],
        "abstract":    row["abstract"],
        "social_post": post,
        "model":       MODEL,
    })

    if (i + 1) % 200 == 0:
        chunk      = pd.DataFrame(results)
        combined   = pd.concat([existing, chunk], ignore_index=True)
        combined.to_parquet(OUTPUT_PATH, index=False)
        existing   = combined
        results    = []
        null_count = chunk["social_post"].isna().sum()
        print(f"\n  ✓ Checkpoint {i+1} — {len(existing)} saved | nulls: {null_count}\n")
        files.download('eval_retrieval_pairs.parquet')

    time.sleep(interval)

# ── FINAL SAVE ────────────────────────────────────────────────────────────────
if results:
    chunk    = pd.DataFrame(results)
    combined = pd.concat([existing, chunk], ignore_index=True)
    combined.to_parquet(OUTPUT_PATH, index=False)

print(f"\nDone. {len(combined)} pairs saved to {OUTPUT_PATH}")
null_total = combined["social_post"].isna().sum()
print(f"Null posts: {null_total} ({null_total/len(combined)*100:.1f}%)")
combined[["domain", "social_post"]].sample(5)

Resuming — 800 rows already done.
Rows to process: 1000 | Model: gpt-oss-120b | 28 RPM
Estimated time: ~36 minutes
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
[10/1000] Turns out P. aeruginosa builds slimey biofilm forts that make it ~10× more resistant to antibiotics 😷. Even crazier, when it teams up with Streptococcus in a mixed biofilm, its nasty factor drops. #microbiology #antibioticresistance 🚀
[20/1000] Just learned that COVID patients have far fewer gut bacteria types and a spike in opportunistic bugs like Streptococcus and Veillonella, unlike flu patients who have a different gut mix. 🤒🦠 #GutHealth #COVID19
[30/1000] New text: "Got word that about 78% of adult ALL patients cleared all detectable disease after just one cycle of blinatum
[40/1000] Did you know >60% of the bugs that make us sick come from animals? Even COVID‑19 probably jumped from bats. Time to push One Health 🐾🌍 #Zoonoses #OneHealth
  [Rate limit] Sleeping 30s...
[50/1000] Did you know that as

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
[210/1000] None
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping

<IPython.core.display.Javascript object>

...

  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
[810/1000] Turns out most college kids say mental‑health apps actually cut their anxiety and depression symptoms and are super easy to use 🙌 #StudentWellness #DigitalTherapy
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
  [Rate limit] Sleeping 30s...
[820/1000] Turns out it's not just the weather—places with poor resource access, weak local governance, and limited cultural knowledge get hit hardest by floods and heatwaves 🌡️💔 #ClimateJustice #Vulnerabil

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. 1800 pairs saved to eval_retrieval_pairs.parquet
Null posts: 175 (9.7%)


,domain,social_post
1278,neuroscience_psychology,Turns out babies don’t just soak up everything...
1159,neuroscience_psychology,"Turns out, eco‑ads work best when they hit all..."
1192,neuroscience_psychology,"Turns out Sgr A*'s shadow is a bright, thick r..."
1265,neuroscience_psychology,Turns out firms that actually build AI capabil...
470,biomedical_health,Just saw that in a huge review of 4.5 M adults...


In [ ]:
models = client.models.list()
for m in models.data:
    print(m.id)

response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say the word hello."}
    ],
    max_tokens=20,
)
print(response)
print("---")
print(response.choices[0].message.content)
print("---")
print(response.choices[0].finish_reason)

gpt-oss-120b
zai-glm-4.7
ChatCompletionResponse(id='chatcmpl-8aa8e3f8-d279-4f4c-913d-fe7b9d86c111', choices=[ChatCompletionResponseChoice(finish_reason='stop', index=0, message=ChatCompletionResponseChoiceMessage(role='assistant', content='hello', reasoning='We just need to comply: "hello".', tool_calls=None), logprobs=None, reasoning_logprobs=None)], created=1779913392, model='gpt-oss-120b', object='chat.completion', system_fingerprint='fp_46358675aa6b1ba4d98d', time_info=ChatCompletionResponseTimeInfo(completion_time=0.012858176, prompt_time=0.002504162, queue_time=0.004005839, total_time=0.020956754684448242, created=1779913392.3242085), usage=ChatCompletionResponseUsage(completion_tokens=20, completion_tokens_details=ChatCompletionResponseUsageCompletionTokensDetails(accepted_prediction_tokens=0, rejected_prediction_tokens=0, reasoning_tokens=9), prompt_tokens=85, prompt_tokens_details=ChatCompletionResponseUsagePromptTokensDetails(cached_tokens=0), total_tokens=105), service_tier=

In [ ]:
import pandas as pd

df1 = pd.read_parquet('eval_retrieval_pairs_first_half.parquet')
df2 = pd.read_parquet('eval_retrieval_pairs_second_half.parquet')

df = pd.concat([df1,df2])
df.to_csv('post_paper_pairs.csv')
